# Network Intrusion Detection with PySpark
**Academic Project — Apache Spark, PySpark, Databricks, Spark MLlib**

Explore **494,021** records from the KDD Cup 99 10% network intrusion benchmark, then train and compare Logistic Regression, Random Forest and Linear SVM models.

**Reproducibility note:** Original coursework outputs are summarized in the repository README. This public notebook omits saved outputs and was not rerun. The historical dataset download URL may be unavailable.

## 1. Environment and Imports

In [ ]:
# Import needed package
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
## For diagram
import matplotlib.pyplot as plt
import seaborn as sns
# Initial Spark
spark = SparkSession.builder.appName("PySparkNetworkIntrusionDetection").getOrCreate()

## 2. Retrieve the KDD Cup 99 Benchmark

Download the 10% KDD Cup 99 sample and move it into Databricks DBFS. The URL is historical; if unavailable, upload the gzip file manually, then skip this cell.

In [ ]:
# Get the dataset
import urllib.request
urllib.request.urlretrieve("http://kdd.ics.uci.edu/databases/kddcup99/kddcup.data_10_percent.gz", "/tmp/kddcup_data.gz")

dbutils.fs.mv("file:/tmp/kddcup_data.gz", "dbfs:/kdd/kddcup_data.gz")
display(dbutils.fs.ls("dbfs:/kdd"))

## 3. Load Network Connections with Spark RDDs

In [ ]:
# Initial RDD
Q3_rdd = spark.sparkContext.textFile("/kdd/kddcup_data.gz")
Q3_rdd.take(10)

In [ ]:
## Verify data type
print("The type of data structure : ", type(Q3_rdd))

## 4. Parse Connection Data and Create a DataFrame

In [ ]:
## The column name
columns = ['Duration', 'Protocol_type', 'Service', 'Flag', 'Src_bytes', 'Dst_bytes', 'Land', 'Wrong_fragment', \
           'Urgent', 'Hot', 'Num_failed_logins', 'Logged_in', 'Num_compromised', 'Root_shell', 'Su_attempted', \
           'Num_root', 'Num_file_creations', 'Num_shells', 'Num_access_files', 'Num_outbound_cmds', \
           'Is_hot_login', 'Is_guest_login', 'Count', 'Serror_rate', 'Rerror_rate', 'Same_srv_rate', \
           'Diff_rv_rate', 'Srv_count', 'Srv_serror_rate', 'Srv_rerror_rate', 'Srv_diff_host_rate', \
           'Dst_host_count', 'Dst_host_srv_count', 'Dst_host_same_srv_rate', 'Dst_host_diff_srv_rate', \
           'Dst_host_same_src_port_rate', 'Dst_host_src_diff_host_rate', 'Dst_host_serror_rate', \
           'Dst_host_srv_serror_rate', 'Dst_host_rerror_rate', 'Dst_host_srv_rerror_rate', 'Label']

## Process the dataset
Q4_rdd = Q3_rdd.map(lambda x: x.split(","))
## Create dataframe
fulldf = Q4_rdd.toDF(columns)

print("Total %i columns." %len(fulldf.columns))
print("\nThe Columns are:\n", fulldf.columns)

## 5. Select Relevant Intrusion Features

In [ ]:
## Select needed columns
df = fulldf[['Duration', 'Protocol_type', 'Service', 'Src_bytes', 'Dst_bytes', 'Flag', 'Label']]
df.show(10)
## Convert to rdd
Q5_rdd = df.rdd
Q5_rdd.take(10)

In [ ]:
df.printSchema()

## 6. Exploratory Analysis: Protocols, Services and Network Traffic

In [ ]:
protocol_df = df.groupBy("Protocol_type").count().sort('count').toPandas()
protocol_df.display()
 
protocol_df.plot.bar(x ='Protocol_type', y='count')
plt.ylabel('count')
plt.title('Group by Protocol_type')
plt.show()

In [ ]:
service_df = df.groupBy("Service").count().sort('count').toPandas()
service_df.display()

service_df.plot.bar(x ='Service', y='count', figsize = (15,10))
plt.ylabel('count')
plt.title('Group by Service')
plt.show()


In [ ]:
temp=df.toPandas().astype({"Src_bytes": int, "Dst_bytes": int, "Duration": int})
temp.head()

In [ ]:
## Get the statistic result for continue features
df.select('Duration').describe().show()
df.select('Src_bytes').describe().show()
df.select('Dst_bytes').describe().show()

temp=df.toPandas().astype({"Src_bytes": int, "Dst_bytes": int, "Duration": int})
## Heatmap
corr = temp.corr('pearson')
sns.heatmap(corr,cmap="Reds", annot=True)
plt.show()
## Stripplot
sns.stripplot(x=temp['Label'],y=temp['Duration'])
plt.xticks(rotation=90)
plt.show()

sns.stripplot(x=temp['Label'],y=temp['Src_bytes'])
plt.xticks(rotation=90)
plt.show()

sns.stripplot(x=temp['Label'],y=temp['Dst_bytes'])
plt.xticks(rotation=90)
plt.show()

In [ ]:
## Data describe for Category features
## Flag
Flag_df = df.groupBy("Flag").count().sort('count').toPandas()
Flag_df.display()

Flag_df.plot.bar(x ='Flag', y='count')
plt.ylabel('count')
plt.title('Group by Flag')
plt.show()
# Label
Label_df=df.groupBy("Label").count().sort('count').toPandas()
Label_df.display()

Label_df.plot.bar(x ='Label', y='count')
plt.ylabel('count')
plt.title('Group by Label')
plt.show()

## 7. Prepare Binary Classification Target

Map `normal.` to class 0 and other labels to class 1; cast numeric fields.

In [ ]:
normal_att_df =df.withColumn("Label_new", when(df["Label"] == 'normal.', 'normal').otherwise('attack'))
normal_att_df.show(10)

In [ ]:
normal_att_df = normal_att_df.withColumn("Duration",col("Duration").cast("int"))
normal_att_df = normal_att_df.withColumn("Src_bytes",col("Src_bytes").cast("int"))
normal_att_df = normal_att_df.withColumn("Dst_bytes",col("Dst_bytes").cast("int"))
normal_att_df.printSchema()

## 8. Encode Categorical Features

**Methodology note:** This original coursework fits StringIndexer before the train/test split. A production-quality evaluation should fit encoders on training data only.

In [ ]:
from pyspark.ml.feature import StringIndexer
## Ref : https://masum-math8065.medium.com/how-to-convert-categorical-data-into-numeric-in-pyspark-2202407f5fac
encoder = StringIndexer(inputCols=["Protocol_type","Service","Flag"], outputCols=["Protocol_type_encoded","Service_encoded","Flag_encoded"]).fit(normal_att_df)
normal_att_df=encoder.transform(normal_att_df)
normal_att_df = normal_att_df.withColumn("Protocol_type_encoded",col("Protocol_type_encoded").cast("int"))
normal_att_df = normal_att_df.withColumn("Service_encoded",col("Service_encoded").cast("int"))
normal_att_df = normal_att_df.withColumn("Flag_encoded",col("Flag_encoded").cast("int"))
normal_att_df.display()
normal_att_df.printSchema()

In [ ]:
normal_att_df = normal_att_df.withColumn("Label_new", when(normal_att_df["Label_new"] == 'normal', int(0)).otherwise(int(1)))
normal_att_df = normal_att_df.select('Duration', 'Src_bytes', 'Dst_bytes', 'Protocol_type_encoded', 'Service_encoded', 'Flag_encoded', 'Label_new')
normal_att_df.display()
normal_att_df.printSchema()

## 9. Assemble Spark MLlib Feature Vectors

In [ ]:
from pyspark.ml.feature import VectorAssembler
# Ref: https://medium.com/@nutanbhogendrasharma/feature-transformer-vectorassembler-in-pyspark-ml-feature-part-3-b3c2c3c93ee9

encoder = VectorAssembler(inputCols = ['Duration', 'Src_bytes', 'Dst_bytes', 'Protocol_type_encoded', 'Service_encoded', 'Flag_encoded'], outputCol="Features")
normal_att_df =encoder.transform(normal_att_df)
full_df = normal_att_df.select(['Features','Label_new'])
full_df.display()

## 10. Split Data into Train and Test Sets

In [ ]:
# Ref: https://medium.com/swlh/logistic-regression-with-pyspark-60295d41221
# Prepare Train and test
train, test = full_df.randomSplit([0.7, 0.3], seed = 2023)
print('The original size: %s'%full_df.count())
print('The size of train set: %s' %train.count())
print('The size of test set: %s' %test.count())

## 11. Train Three Spark MLlib Classifiers

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, LinearSVC

## Prepare two model
lr = LogisticRegression(featuresCol = 'Features', labelCol = 'Label_new')
rf = RandomForestClassifier(featuresCol = 'Features', labelCol = 'Label_new')
lsvc = LinearSVC(featuresCol = 'Features', labelCol = 'Label_new')

model1=lr.fit(train)
model2=rf.fit(train)
model3=lsvc.fit(train)

## 12. Evaluate Logistic Regression, Random Forest, and Linear SVM

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
## Evaluate for model 1
predictions = model1.transform(test)
predictions = predictions.select(['prediction', 'Label_new'])
predictions.show(2)

eva_accuracy = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='accuracy')
eva_precision = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction',metricName='precisionByLabel')
eva_recall = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='recallByLabel')
eva_true_positive = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='truePositiveRateByLabel')
eva_f1 = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='f1')
 

accuracy = eva_accuracy.evaluate(predictions)
precision = eva_precision.evaluate(predictions)
recall=eva_recall.evaluate(predictions)
true_positive = eva_true_positive.evaluate(predictions)
f1 = eva_f1.evaluate(predictions)

print('Accuracy: ', accuracy)
print('Precision: ', precision)
print('Recall: ', recall)
print('True positive rate: ', true_positive)
print('F1 score: ', f1)

In [ ]:
## Evaluate for model 2
predictions = model2.transform(test)
predictions = predictions.select(['prediction', 'Label_new'])
predictions.show(2)

eva_accuracy = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='accuracy')
eva_precision = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction',metricName='precisionByLabel')
eva_recall = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='recallByLabel')
eva_true_positive = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='truePositiveRateByLabel')
eva_f1 = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='f1')
 

accuracy = eva_accuracy.evaluate(predictions)
precision = eva_precision.evaluate(predictions)
recall=eva_recall.evaluate(predictions)
true_positive = eva_true_positive.evaluate(predictions)
f1 = eva_f1.evaluate(predictions)

print('Accuracy: ', accuracy)
print('Precision: ', precision)
print('Recall: ', recall)
print('True positive rate: ', true_positive)
print('F1 score: ', f1)

## 12. Evaluate Logistic Regression, Random Forest, and Linear SVM

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
## Evaluate for model 1
predictions = model3.transform(test)
predictions = predictions.select(['prediction', 'Label_new'])
predictions.show(2)

eva_accuracy = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='accuracy')
eva_precision = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction',metricName='precisionByLabel')
eva_recall = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='recallByLabel')
eva_true_positive = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='truePositiveRateByLabel')
eva_f1 = MulticlassClassificationEvaluator(labelCol='Label_new', predictionCol='prediction', metricName='f1')
 

accuracy = eva_accuracy.evaluate(predictions)
precision = eva_precision.evaluate(predictions)
recall=eva_recall.evaluate(predictions)
true_positive = eva_true_positive.evaluate(predictions)
f1 = eva_f1.evaluate(predictions)

print('Accuracy: ', accuracy)
print('Precision: ', precision)
print('Recall: ', recall)
print('True positive rate: ', true_positive)
print('F1 score: ', f1)